# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [5]:

# 1. Déclarer les tables Bronze
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/order_items",
    "secondary": "/lakehouse/bronze/orders"
}

# 2. Charger les tables avec Spark
bronze_dfs = {
    alias: spark.read.format("delta").load(path)
    for alias, path in BRONZE_TABLES.items()
}

# 3. Afficher les informations de chaque table
for alias, df in bronze_dfs.items():
    print(
        f"[{alias}] {BRONZE_TABLES[alias]} "
        f"-> {df.count():,} lignes | "
        f"colonnes : {df.columns}"
    )

# 4. Sélectionner la table principale
df_bronze = bronze_dfs["main"]

# 5. Compter les valeurs NULL de chaque colonne
null_report = df_bronze.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])

print("Diagnostic des valeurs NULL dans order_items :")
null_report.show(truncate=False)

# 6. Vérifier les doublons de la table principale
count_total = df_bronze.count()

count_distinct = df_bronze.dropDuplicates(
    ["order_id", "order_item_id"]
).count()

print("Nombre total de lignes :", count_total)
print("Nombre de doublons :", count_total - count_distinct)


26/11/17 10:50:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

[main] /lakehouse/bronze/order_items -> 112,650 lignes | colonnes : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', '_ingested_at', '_source_dataset', '_source_file']


[secondary] /lakehouse/bronze/orders -> 99,441 lignes | colonnes : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_ingested_at', '_source_dataset', '_source_file']
Diagnostic des valeurs NULL dans order_items :


+--------+-------------+----------+---------+-------------------+-----+-------------+------------+---------------+------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|_ingested_at|_source_dataset|_source_file|
+--------+-------------+----------+---------+-------------------+-----+-------------+------------+---------------+------------+
|0       |0            |0         |0        |0                  |0    |0            |0           |0              |0           |
+--------+-------------+----------+---------+-------------------+-----+-------------+------------+---------------+------------+



Nombre total de lignes : 112650
Nombre de doublons : 0


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [6]:

from pyspark.sql import functions as F

# 1. Définir le chemin de la table Silver
SILVER_MAIN_PATH = "/lakehouse/silver/olist_order_items_enriched"

# 2. Récupérer les tables Bronze
df_items = bronze_dfs["main"]
df_orders = bronze_dfs["secondary"]

# 3. Nettoyer la table principale
df_items_clean = (
    df_items
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_item_id").isNotNull())
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
)

# Nombre de lignes avant la jointure
count_before_join = df_items_clean.count()

# 4. Préparer la table secondaire
df_orders_clean = (
    df_orders
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    )
)

# 5. Effectuer la jointure LEFT
df_silver = (
    df_items_clean
    .join(df_orders_clean, on="order_id", how="left")
)

# 6. Ajouter deux colonnes calculées
df_silver = (
    df_silver
    .withColumn(
        "item_total_amount",
        F.col("price") + F.col("freight_value")
    )
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
)

# 7. Afficher le plan d'exécution
print("Plan d'exécution Spark :")
df_silver.explain("simple")

# 8. Enregistrer dans Delta Lake
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

# 9. Vérifier le résultat
silver_count = (
    spark.read.format("delta")
    .load(SILVER_MAIN_PATH)
    .count()
)

assert silver_count <= count_before_join, (
    f"Explosion N-M détectée : {silver_count} > {count_before_join}"
)

print("Lignes avant jointure :", count_before_join)
print("Lignes après jointure :", silver_count)
print("Table Silver enregistrée :", SILVER_MAIN_PATH)


Plan d'exécution Spark :


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#0, order_item_id#1, product_id#1674, seller_id#1676, shipping_limit_date#1678, price#1564, freight_value#1565, _ingested_at#1684, _source_dataset#1686, _source_file#1688, customer_id#1690, order_status#1692, order_purchase_timestamp#1694, order_delivered_customer_date#1700, (price#1564 + freight_value#1565) AS item_total_amount#1671, datediff(cast(order_delivered_customer_date#1700 as date), cast(order_purchase_timestamp#1694 as date)) AS delivery_delay_days#1672]
   +- BroadcastHashJoin [order_id#0], [order_id#10], LeftOuter, BuildRight, false, false
      :- SortAggregate(key=[order_id#0, order_item_id#1], functions=[first(product_id#2, false), first(seller_id#3, false), first(shipping_limit_date#4, false), first(price#5, false), first(freight_value#6, false), first(_ingested_at#7, false), first(_source_dataset#8, false), first(_source_file#9, false)])
      :  +- Sort [order_id#0 ASC NULLS FIRST, order_item

[Stage 61:====================================================>   (47 + 3) / 50]

Lignes avant jointure : 112650
Lignes après jointure : 112650
Table Silver enregistrée : /lakehouse/silver/olist_order_items_enriched


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [11]:
# 1. Lire le journal transactionnel Delta
print("=== JOURNAL DELTA ===")

spark.read.text(
    f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json"
).show(8, truncate=110)


# 2. Créer un DataFrame volontairement incompatible
print("=== DATAFRAME INCOMPATIBLE ===")

bad_df = spark.createDataFrame(
    [("test_1", "colonne_intruse")],
    ["fake_id", "intruder_col"]
)

bad_df.printSchema()


# 3. Tester le Schema Enforcement
print("=== TEST SCHEMA ENFORCEMENT ===")

try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)

    print("ATTENTION : écriture acceptée, résultat inattendu.")

except Exception as e:
    print("Schema Enforcement validé : écriture invalide bloquée.")
    print("Erreur :", str(e).splitlines()[0])

=== JOURNAL DELTA ===
+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794912877999,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"75cac158-61c4-44fc-8329-b1ec37995922","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-d9bd21ec-f63d-4fdd-8953-eb9a55eea31e-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-b38d2574-159a-4e29-983a-8380f43792ba-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-8152dbf1-ae84-4e8e-8457-61c22aabb49e-c000.snap

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [15]:
# TODO : Exécuter un MERGE INTO sur `SILVER_MAIN_PATH` puis comparer versionAsOf=0 et la version courante
target_delta = DeltaTable.forPath(spark, MERGE_TEST_PATH)

(
    target_delta.alias("target")
    .merge(
        source_df.alias("source"),
        """
        target.order_id = source.order_id
        AND target.order_item_id = source.order_item_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("MERGE INTO exécuté avec succès !")

MERGE INTO exécuté avec succès !


26/11/17 11:08:50 WARN MapPartitionsRDD: RDD 284 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [2]:
print("Port Spark UI :", spark.sparkContext.uiWebUrl)

Port Spark UI : http://spark-jupyter:4041


In [3]:
print("Version Spark :", spark.version)

bronze_path = "hdfs://namenode:9000/lakehouse/bronze"

for item in spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
).listStatus(
    spark._jvm.org.apache.hadoop.fs.Path(bronze_path)
):
    print(item.getPath().getName())

Version Spark : 4.2.0
customers
geolocation
order_items
order_payments
order_reviews
orders
product_category_translation
products
raw
sellers


In [5]:

# Vérifier les valeurs NULL sur order_id dans les deux tables

for alias, df in bronze_dfs.items():
    null_count = df.filter(
        F.col("order_id").isNull()
    ).count()

    print(f"{alias} : {null_count} valeurs NULL dans order_id")


main : 0 valeurs NULL dans order_id


[Stage 40:==================================================>     (45 + 4) / 50]

secondary : 0 valeurs NULL dans order_id


In [6]:
df_orders = bronze_dfs["secondary"]

total_orders = df_orders.count()

distinct_orders = df_orders.dropDuplicates(
    ["order_id"]
).count()

print("Nombre total de commandes :", total_orders)
print("Nombre de commandes uniques :", distinct_orders)
print("Nombre de doublons :", total_orders - distinct_orders)

[Stage 55:==========================================================(4 + 0) / 4]

Nombre total de commandes : 99441
Nombre de commandes uniques : 99441
Nombre de doublons : 0


In [7]:
df_check = (
    spark.read
    .format("delta")
    .load("/lakehouse/silver/olist_order_items_enriched")
)

print("Nombre de lignes :", df_check.count())
print("Nombre de colonnes :", len(df_check.columns))

df_check.printSchema()

df_check.show(5, truncate=False)

Nombre de lignes : 112650
Nombre de colonnes : 16
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_dataset: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- item_total_amount: double (nullable = true)
 |-- delivery_delay_days: integer (nullable = true)



+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+-------------------+-------------------------------------------------------------------------------------------+--------------------------------+------------+------------------------+-----------------------------+------------------+-------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at              |_source_dataset    |_source_file                                                                               |customer_id                     |order_status|order_purchase_timestamp|order_delivered_customer_date|item_total_amount |delivery_delay_days|
+--------------------------------+-------------+--------------------------------+--------------------------------+------------------

In [8]:
from delta.tables import DeltaTable

SILVER_MAIN_PATH = "/lakehouse/silver/olist_order_items_enriched"

delta_table = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

delta_table.history().select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                               |
+-------+-----------------------+---------+------------------------------------------------------------------+
|0      |2026-11-17 10:54:39.543|WRITE    |{mode -> Overwrite, partitionBy -> [], canOverwriteSchema -> true}|
+-------+-----------------------+---------+------------------------------------------------------------------+



In [10]:
df_old = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(SILVER_MAIN_PATH)
)

print("Nombre de lignes version 0 :", df_old.count())

df_old.show(5, truncate=False)

Nombre de lignes version 0 : 112650


+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+-------------------+-------------------------------------------------------------------------------------------+--------------------------------+------------+------------------------+-----------------------------+------------------+-------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at              |_source_dataset    |_source_file                                                                               |customer_id                     |order_status|order_purchase_timestamp|order_delivered_customer_date|item_total_amount |delivery_delay_days|
+--------------------------------+-------------+--------------------------------+--------------------------------+------------------

In [12]:
df_final_check = (
    spark.read
    .format("delta")
    .load(SILVER_MAIN_PATH)
)

print("Nombre de lignes après le test :", df_final_check.count())
print("Nombre de colonnes :", len(df_final_check.columns))

Nombre de lignes après le test : 112650
Nombre de colonnes : 16


In [13]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

MERGE_TEST_PATH = "/lakehouse/silver/olist_order_items_merge_test"

# Copier la table Silver dans une table de test
(
    spark.read.format("delta")
    .load(SILVER_MAIN_PATH)
    .write.format("delta")
    .mode("overwrite")
    .save(MERGE_TEST_PATH)
)

print("Copie créée :", MERGE_TEST_PATH)
print(
    "Nombre de lignes :",
    spark.read.format("delta").load(MERGE_TEST_PATH).count()
)


Copie créée : /lakehouse/silver/olist_order_items_merge_test


[Stage 108:===============================================>       (43 + 4) / 50]

Nombre de lignes : 112650


In [14]:
# Lire la copie
df_target = spark.read.format("delta").load(MERGE_TEST_PATH)

# Prélever une ligne existante
existing_row = df_target.first().asDict()

# Créer une ligne de test à partir de cette ligne
new_row = existing_row.copy()
new_row["order_id"] = "MERGE_TEST_ORDER_001"
new_row["_source_dataset"] = "merge_test"
new_row["_source_file"] = "manual_test"

# Construire le DataFrame source
source_df = spark.createDataFrame(
    [existing_row, new_row],
    schema=df_target.schema
)

print("Données à fusionner :")
source_df.select("order_id", "order_item_id", "order_status").show()

Données à fusionner :


[Stage 121:===================>                                     (1 + 2) / 3]

+--------------------+-------------+------------+
|            order_id|order_item_id|order_status|
+--------------------+-------------+------------+
|00018f77f2f0320c5...|            1|   delivered|
|MERGE_TEST_ORDER_001|            1|   delivered|
+--------------------+-------------+------------+



In [16]:
history_df = spark.sql(
    f"DESCRIBE HISTORY delta.`{MERGE_TEST_PATH}`"
)

history_df.select(
    "version",
    "timestamp",
    "operation",
    "operationMetrics"
).show(truncate=False)

+-------+-----------------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                         

In [17]:
# Version initiale
df_version_0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(MERGE_TEST_PATH)
)

# Version actuelle
df_current = (
    spark.read.format("delta")
    .load(MERGE_TEST_PATH)
)

print("Lignes version 0 :", df_version_0.count())
print("Lignes version actuelle :", df_current.count())

print("Ligne de test insérée :")
df_current.filter(
    F.col("order_id") == "MERGE_TEST_ORDER_001"
).show(truncate=False)

Lignes version 0 : 112650


Lignes version actuelle : 112651
Ligne de test insérée :


+--------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+---------------+------------+--------------------------------+------------+------------------------+-----------------------------+-----------------+-------------------+
|order_id            |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|_ingested_at              |_source_dataset|_source_file|customer_id                     |order_status|order_purchase_timestamp|order_delivered_customer_date|item_total_amount|delivery_delay_days|
+--------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+--------------------------+---------------+------------+--------------------------------+------------+------------------------+-----------------------------+-----------------+-------